In [0]:
from pyspark.sql.functions import col, to_date, from_unixtime 

In [0]:
# Configuration 

source_table = "dbacademy.healthcare.customer" 

target_catalog = "dbacademy" 

target_schema = "healthcare" 

target_table = f"{target_catalog}.{target_schema}.bronze_customers" 

 

In [0]:
# Read source data 
df = spark.read.table(source_table) 

# Convert data types: 
# customer_id -> LONG (already LONG, cast to ensure) 
# valid_from -> DATE (from epoch seconds) 
# valid_to -> DATE (from string epoch seconds) 
# units_purchased -> numeric (already LONG, cast to ensure) 
from pyspark.sql.functions import when

df_cleaned = df.select( 
    col("customer_id").cast("long"), col("tax_id"), col("tax_code"), col("customer_name"), 
    col("state"), col("city"), col("postcode"), col("street"), col("number"), col("unit"), 
    col("region"), col("district"), col("lon"), col("lat"), col("ship_to_address"), 
    to_date(from_unixtime(col("valid_from"))).alias("valid_from"), 
    to_date(from_unixtime(
        when(col("valid_to") == "NULL", None)
         .otherwise(col("valid_to").cast("long"))
    )).alias("valid_to"), 
    col("units_purchased").cast("long"),  col("loyalty_segment") 
) 

In [0]:
 # Save as bronze_customers table 
df_cleaned.write.mode("overwrite").saveAsTable(target_table) 


In [0]:
print(f"Successfully saved bronze_customers to {target_table}") 

print(f"Record count: {spark.read.table(target_table).count()}") 
